# W1 종합: 이미지 기초 통합 파이프라인

> 📖 원리 이해: [LearnOpenCV - Image Processing](https://learnopencv.com/image-processing-in-opencv/)
> 📋 파라미터 확인: [OpenCV 공식 - Core Operations](https://docs.opencv.org/4.x/d7/d16/tutorial_py_table_of_contents_core.html)

---

## 📌 오늘의 목표
- W1D1~D5에서 배운 함수를 **하나의 파이프라인**으로 연결하기
- NEU 6종 결함 전체에 파이프라인 적용해 결과 비교하기
- 각 단계가 왜 필요한지 다시 확인하기

## 🎯 오늘 쓰는 함수 (W1 전체 복습)

| 날 | 함수 | 역할 |
|----|------|------|
| D1 | `cv2.imread`, `shape`, `dtype` | 이미지 로딩 + 기본 정보 |
| D2 | `cv2.cvtColor`, ROI slicing | 색공간 변환 + 관심 영역 |
| D3 | `cv2.calcHist` | 픽셀 분포 분석 |
| D4 | `cv2.createCLAHE` | 대비 향상 전처리 |
| D5 | `cv2.absdiff`, `cv2.bitwise_and`, `cv2.addWeighted` | 차이 감지 + 결함 추출 |

## 📦 Import + 데이터 로딩

오늘은 NEU 6종 결함 각각에서 두 장씩 로딩합니다. 파이프라인을 6종 전체에 적용해 결과를 비교합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path('../data/kaggle/archive/NEU-DET/train/images')
DEFECTS  = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']

def load_pair(defect):
    ref  = cv2.imread(str(DATA_DIR / defect / f'{defect}_1.jpg'), cv2.IMREAD_GRAYSCALE)
    test = cv2.imread(str(DATA_DIR / defect / f'{defect}_2.jpg'), cv2.IMREAD_GRAYSCALE)
    return ref, test

ref, test = load_pair('inclusion')
print(f'shape: {ref.shape}, dtype: {ref.dtype}')
print(f'픽셀 범위: {ref.min()} ~ {ref.max()}')

## 🔧 1단계: 이미지 기본 정보 확인 (D1 복습)

**이 코드가 하는 것:** 두 이미지의 shape, dtype, 픽셀 범위를 확인합니다.

**왜 먼저 확인하나:** 두 이미지의 shape이나 dtype이 다르면 absdiff, addWeighted 등 모든 연산이 오류납니다. 파이프라인 진입 전 체크가 필수입니다.

**제조 검사 연결:** 실제 AOI에서도 카메라 설정(해상도, 비트심도)이 기준 이미지와 다르면 비교 자체가 불가합니다.

In [ ]:
def check_image_info(img_ref, img_test, defect_name):
    compatible = (img_ref.shape == img_test.shape) and (img_ref.dtype == img_test.dtype)
    print(f'[{defect_name}]')
    print(f'  참조: shape={img_ref.shape}, dtype={img_ref.dtype}, 범위={img_ref.min()}~{img_ref.max()}')
    print(f'  검사: shape={img_test.shape}, dtype={img_test.dtype}, 범위={img_test.min()}~{img_test.max()}')
    print(f'  호환 여부: {"✅ OK" if compatible else "❌ 불일치 — 파이프라인 중단 필요"}')
    return compatible

check_image_info(ref, test, 'inclusion')

### 해석 가이드
- shape, dtype이 동일해야 이후 모든 연산이 작동합니다
- 픽셀 범위가 0~255면 uint8 정상 로딩

> 🤔 **판단 질문 1:** shape은 같은데 dtype이 다른 경우(uint8 vs float32)가 생기는 상황은 어떤 경우일까요?

---

### 🔨 깨뜨리기 챌린지
```
- 챌린지 1: IMREAD_COLOR로 로딩한 이미지와 IMREAD_GRAYSCALE 이미지를 absdiff에 넣으면 어떤 오류가 날까요?
- 챌린지 2: img_ref.astype(np.float32)로 dtype을 바꾼 뒤 check_image_info를 실행하면 호환 여부가 어떻게 바뀌나요?
```
예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 2단계: ROI 설정 (D2 복습)

**이 코드가 하는 것:** 이미지 중앙 부분만 잘라내 관심 영역(ROI)을 설정합니다.

**왜 ROI를 쓰나:** 전체 이미지 비교 시 외곽 경계(조명 그라데이션, 프레임 노이즈)가 결함으로 오검될 수 있습니다. 실제 검사 영역만 잘라내면 정확도가 올라갑니다.

**제조 검사 연결:** SPI 검사에서도 기판 외곽을 제외하고 패드 영역만 ROI로 설정한 뒤 비교하는 방식이 기본입니다.

In [ ]:
def apply_roi(img, margin=20):
    h, w = img.shape
    return img[margin:h-margin, margin:w-margin]

ref_roi  = apply_roi(ref)
test_roi = apply_roi(test)

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(ref,      cmap='gray'); axes[0].set_title('참조 원본');   axes[0].axis('off')
axes[1].imshow(ref_roi,  cmap='gray'); axes[1].set_title('참조 ROI');    axes[1].axis('off')
axes[2].imshow(test,     cmap='gray'); axes[2].set_title('검사 원본');   axes[2].axis('off')
axes[3].imshow(test_roi, cmap='gray'); axes[3].set_title('검사 ROI');    axes[3].axis('off')
plt.suptitle('ROI 설정: 외곽 margin=20 제거', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'원본: {ref.shape} → ROI: {ref_roi.shape}')

### 해석 가이드
- 외곽 20픽셀이 제거되어 이미지가 약간 작아집니다
- 경계 밝기 변화가 있는 이미지라면 ROI 적용 전후 차이가 뚜렷합니다

> 🤔 **판단 질문 1:** margin을 너무 크게(예: 80픽셀) 잡으면 어떤 문제가 생길까요?

---

### 🔨 깨뜨리기 챌린지
```
- 챌린지 1: margin=80으로 변경하면 ROI shape은 얼마가 될까요? 이미지가 너무 작아지는 기준은?
- 챌린지 2: 중앙이 아니라 왼쪽 절반만 ROI로 설정하려면 슬라이싱을 어떻게 바꿔야 할까요?
```
예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 3단계: 히스토그램 분석 (D3 복습)

**이 코드가 하는 것:** CLAHE 적용 전후 히스토그램을 비교해 대비 향상 효과를 수치로 확인합니다.

**왜 히스토그램을 보나:** 픽셀 분포가 좁게 몰려 있으면(저대비) 결함과 배경의 구분이 어렵습니다. CLAHE 전후 히스토그램을 비교해 전처리 효과를 검증합니다.

**제조 검사 연결:** 조명 조건이 달라졌을 때 히스토그램 분포로 이미지 품질을 자동 판별합니다.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
ref_clahe  = clahe.apply(ref_roi)
test_clahe = clahe.apply(test_roi)

hist_before = cv2.calcHist([ref_roi],   [0], None, [256], [0, 256])
hist_after  = cv2.calcHist([ref_clahe], [0], None, [256], [0, 256])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_before, color='steelblue', linewidth=1.5)
axes[0].set_title('CLAHE 적용 전 히스토그램', fontsize=12)
axes[0].set_xlabel('픽셀값'); axes[0].set_ylabel('픽셀 수')
axes[0].grid(alpha=0.3)

axes[1].plot(hist_after, color='tomato', linewidth=1.5)
axes[1].set_title('CLAHE 적용 후 히스토그램', fontsize=12)
axes[1].set_xlabel('픽셀값'); axes[1].set_ylabel('픽셀 수')
axes[1].grid(alpha=0.3)

plt.suptitle('히스토그램 비교: CLAHE 전후', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 해석 가이드
- **적용 전**: 픽셀이 특정 구간에 몰려 있으면 저대비 이미지
- **적용 후**: 분포가 넓게 퍼지면 CLAHE가 대비를 향상시킨 것
- 피크가 지나치게 높으면 조명 과노출 or 과소노출 가능성

> 🤔 **판단 질문 1:** 히스토그램 적용 후에도 분포가 거의 변하지 않는다면, 원본 이미지의 대비가 이미 충분하다는 뜻일까요 아니면 clipLimit가 너무 낮다는 뜻일까요?

---

### 🔨 깨뜨리기 챌린지
```
- 챌린지 1: clipLimit=0.5로 낮추면 히스토그램이 CLAHE 전과 거의 같아질까요?
- 챌린지 2: tileGridSize=(2, 2)로 바꾸면 히스토그램 모양이 어떻게 달라질까요?
```
예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 4단계: CLAHE 전처리 시각화 (D4 복습)

**이 코드가 하는 것:** CLAHE 적용 전후 이미지를 나란히 비교합니다.

**왜 이 단계가 있나:** 히스토그램은 분포만 보여주지만, 실제 이미지를 눈으로 확인해야 전처리가 결함 가시성을 높였는지 직접 검증할 수 있습니다.

**제조 검사 연결:** 전처리 파라미터 튜닝 시 수치(히스토그램)와 시각 확인을 같이 하는 것이 실무 표준입니다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(ref_roi,   cmap='gray'); axes[0, 0].set_title('참조 — CLAHE 전'); axes[0, 0].axis('off')
axes[0, 1].imshow(ref_clahe, cmap='gray'); axes[0, 1].set_title('참조 — CLAHE 후'); axes[0, 1].axis('off')
axes[1, 0].imshow(test_roi,   cmap='gray'); axes[1, 0].set_title('검사 — CLAHE 전'); axes[1, 0].axis('off')
axes[1, 1].imshow(test_clahe, cmap='gray'); axes[1, 1].set_title('검사 — CLAHE 후'); axes[1, 1].axis('off')

plt.suptitle('CLAHE 전처리 시각화 (inclusion)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 해석 가이드
- CLAHE 후 결함 경계가 더 선명하게 보이면 전처리 성공
- 배경 노이즈도 같이 강조되면 clipLimit을 낮춰야 합니다

> 🤔 **판단 질문 1:** CLAHE 적용 후 결함이 더 잘 보이는 결함 유형과 그렇지 않은 유형이 있다면, 그 차이는 어디서 올까요?

---

### 🔨 깨뜨리기 챌린지
```
- 챌린지 1: clipLimit=8.0으로 올리면 이미지에서 어떤 변화가 생기나요? 노이즈와 결함 중 어느 것이 더 강조될까요?
- 챌린지 2: CLAHE 대신 equalizeHist를 써보세요. 결과가 더 좋나요 나쁜가요?
```
예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 5단계: absdiff + bitwise 결함 추출 (D5 복습)

**이 코드가 하는 것:** CLAHE 전처리된 두 이미지를 absdiff로 비교하고, threshold + bitwise_and로 결함 영역만 추출합니다.

**왜 CLAHE 후에 absdiff를 하나:** 전처리 없이 비교하면 조명 차이로 생긴 가짜 차이가 결함으로 잡힙니다. CLAHE로 두 이미지의 대비를 먼저 균일하게 맞추면 실제 결함 차이가 더 잘 드러납니다.

**왜 이 방법인가 (대안과 트레이드오프):**
- 단순 픽셀 비교(absdiff)는 빠르고 해석이 쉽지만, 두 이미지가 정렬되지 않으면 오검이 많습니다
- 정렬이 보장된 데이터에서는 이 방법이 가장 단순하고 빠릅니다 (W6에서 정렬 문제 해결)

In [ ]:
diff      = cv2.absdiff(ref_clahe, test_clahe)
_, mask   = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
extracted = cv2.bitwise_and(test_clahe, test_clahe, mask=mask)

diff_color   = cv2.applyColorMap(cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX), cv2.COLORMAP_HOT)
test_bgr     = cv2.cvtColor(test_clahe, cv2.COLOR_GRAY2BGR)
overlay      = cv2.addWeighted(test_bgr, 0.6, diff_color, 0.4, 0)

defect_ratio = mask.sum() // 255 / mask.size * 100

fig, axes = plt.subplots(1, 5, figsize=(28, 6))
panels = [
    (ref_clahe,  'gray', '참조 (CLAHE 후)'),
    (test_clahe, 'gray', '검사 (CLAHE 후)'),
    (diff,       'hot',  f'absdiff\nmax={diff.max()}'),
    (mask,       'gray', f'마스크(>30)\n결함 {defect_ratio:.1f}%'),
    (extracted,  'gray', '결함 추출'),
]
for ax, (im, cmap, title) in zip(axes, panels):
    ax.imshow(im, cmap=cmap, vmin=0, vmax=255); ax.set_title(title, fontsize=11); ax.axis('off')

plt.suptitle('5단계: absdiff → threshold → bitwise_and (inclusion)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 해석 가이드
- **diff**: 밝을수록 두 이미지가 다른 픽셀 → 결함 후보
- **마스크**: 흰색이 결함으로 판정된 영역. 비율이 너무 높으면(>30%) 임계값이 낮거나 두 이미지 자체가 많이 다른 것
- **결함 추출**: 마스크 영역만 남기고 나머지는 검정. 결함 위치를 바로 확인 가능

> 🤔 **판단 질문 1:** CLAHE 적용 전 이미지로 absdiff를 했을 때와 후로 했을 때 결함 픽셀 비율이 달라지나요? 더 정확한 쪽은 어느 쪽일까요?

> 🤔 **판단 질문 2:** threshold를 30에서 10으로 낮추면 결함 비율이 올라가는 건 좋은 걸까요 나쁜 걸까요?

---

### 🔨 깨뜨리기 챌린지
```
- 챌린지 1: CLAHE 전 이미지(ref_roi, test_roi)로 absdiff를 해보세요. 결함 픽셀 비율이 달라지나요?
- 챌린지 2: threshold를 10으로 낮추고 50으로 올려보세요. 어느 쪽이 이 결함 유형에 더 적합한가요?
- 챌린지 3: mask 대신 cv2.bitwise_not(mask)를 bitwise_and에 넣으면 무엇이 추출될까요?
```
예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 6단계: W1 통합 파이프라인 — NEU 6종 전체 적용

**이 코드가 하는 것:** W1D1~D5의 모든 단계를 함수로 묶어 6종 결함 전체에 적용합니다.

**왜 함수로 묶나:** 6종에 같은 처리를 반복하려면 파이프라인이 함수여야 합니다. 나중에 파라미터만 바꿔 전체를 재실행하는 것도 가능해집니다.

**제조 검사 연결:** 실제 검사 소프트웨어도 이런 파이프라인 구조입니다. 각 단계를 모듈화해두면 특정 결함 유형에만 파라미터를 다르게 적용할 수 있습니다.

In [ ]:
def w1_pipeline(img_ref, img_test, margin=20, clip_limit=2.0, threshold_val=30, alpha=0.6):
    roi_ref  = img_ref[margin:-margin, margin:-margin]
    roi_test = img_test[margin:-margin, margin:-margin]

    clahe      = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    enh_ref    = clahe.apply(roi_ref)
    enh_test   = clahe.apply(roi_test)

    diff       = cv2.absdiff(enh_ref, enh_test)
    _, mask    = cv2.threshold(diff, threshold_val, 255, cv2.THRESH_BINARY)
    extracted  = cv2.bitwise_and(enh_test, enh_test, mask=mask)

    diff_color = cv2.applyColorMap(cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX), cv2.COLORMAP_HOT)
    test_bgr   = cv2.cvtColor(enh_test, cv2.COLOR_GRAY2BGR)
    overlay    = cv2.addWeighted(test_bgr, alpha, diff_color, 1 - alpha, 0)

    ratio = mask.sum() // 255 / mask.size * 100
    return enh_ref, enh_test, diff, mask, extracted, overlay, ratio


fig, axes = plt.subplots(len(DEFECTS), 5, figsize=(28, len(DEFECTS) * 5))
col_titles = ['참조 (CLAHE)', '검사 (CLAHE)', 'absdiff', '마스크', '오버레이']
for col, t in enumerate(col_titles):
    axes[0, col].set_title(t, fontsize=13, fontweight='bold', pad=10)

ratios = {}
for row, defect in enumerate(DEFECTS):
    r, t = load_pair(defect)
    enh_ref, enh_test, diff, mask, extracted, overlay, ratio = w1_pipeline(r, t)
    ratios[defect] = ratio

    axes[row, 0].imshow(enh_ref,   cmap='gray'); axes[row, 0].set_ylabel(defect.replace('_', '\n'), fontsize=10, fontweight='bold'); axes[row, 0].axis('off')
    axes[row, 1].imshow(enh_test,  cmap='gray'); axes[row, 1].axis('off')
    axes[row, 2].imshow(diff,      cmap='hot',  vmin=0, vmax=255); axes[row, 2].axis('off')
    axes[row, 3].imshow(mask,      cmap='gray'); axes[row, 3].set_xlabel(f'{ratio:.1f}%', fontsize=10); axes[row, 3].axis('off')
    axes[row, 4].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[row, 4].axis('off')

plt.suptitle('W1 통합 파이프라인: NEU 6종 결함 전체 적용', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n📊 결함 유형별 검출 픽셀 비율 (threshold=30):')
for d, r in sorted(ratios.items(), key=lambda x: -x[1]):
    print(f'  {d:20s}: {r:.2f}%')

### 해석 가이드
- **결함 비율 높은 유형**: 두 이미지 간 결함 위치 차이가 크거나, 결함이 넓게 분포하는 유형
- **결함 비율 낮은 유형**: 결함이 작고 국소적이거나, 두 이미지가 우연히 비슷한 위치에 결함이 있는 경우
- 오버레이에서 빨간/주황 부분이 실제 결함과 일치하는지 확인해보세요

> 🤔 **판단 질문 1:** 6종 중 어느 결함이 이 파이프라인으로 가장 잘 검출되고, 어느 결함이 가장 안 될까요? 이유는?

> 🤔 **판단 질문 2:** 이 파이프라인의 가장 큰 약점은 무엇이고, W6에서 어떻게 보완될까요?

---

### 🔨 깨뜨리기 챌린지
```
- 챌린지 1: threshold_val=10으로 낮추면 6종 전체의 결함 비율 순위가 바뀌나요?
- 챌린지 2: clip_limit=0.5로 낮추면 어떤 결함 유형에서 변화가 가장 클까요?
- 챌린지 3: margin=0으로 하면(ROI 제거) 결함 비율이 올라가나요 내려가나요? 왜?
```
예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 📝 W1 정리

### W1에서 배운 것

| 단계 | 함수 | 역할 | 파이프라인에서 위치 |
|------|------|------|--------------------|
| 로딩 | `imread`, `shape`, `dtype` | 이미지 입력 + 호환 확인 | 시작 |
| ROI | 슬라이싱 `[y:h, x:w]` | 관심 영역만 잘라내기 | 전처리 1 |
| 대비 | `createCLAHE` | 결함 가시성 향상 | 전처리 2 |
| 분석 | `calcHist` | 픽셀 분포 확인 | 검증 |
| 비교 | `absdiff` | 두 이미지 차이 계산 | 핵심 연산 |
| 추출 | `threshold` + `bitwise_and` | 결함 영역만 분리 | 후처리 |
| 시각화 | `addWeighted` | 결과 오버레이 | 출력 |

### W1의 한계 (W2~W6에서 해결)
```
지금의 문제          →    해결하는 주차
노이즈를 결함으로 잡음  →    W2 (blur 전처리)
결함 영역이 끊겨 보임  →    W3 (morphology)
결함 크기/위치 측정 없음 →    W4 (contour)
두 이미지 정렬 안 됨   →    W5/W6 (변환 + feature matching)
```

### 복습 퀴즈
1. `cv2.absdiff(a, b)`와 `a - b` (numpy)의 차이는? uint8에서 50 - 200의 결과는 각각 얼마?
2. CLAHE의 `clipLimit`을 높이면 어떤 효과가 생기고, 너무 높으면 어떤 문제가 있나요?
3. `bitwise_and(img, img, mask=mask)`에서 mask 픽셀이 0인 위치의 결과값은?
4. `addWeighted`에서 `alpha + beta > 1.0`이면 결과 이미지가 어떻게 되나요?
5. 이 파이프라인을 실제 라인에 적용할 때 가장 먼저 해결해야 할 문제는 무엇일까요?